# Conversation Memory with LiteLLM
This notebook demonstrates implementing conversation memory techniques using LiteLLM for LLM calls. Since LiteLLM is a unified API without built-in memory, we implement memory manually by managing message history.

**Advantages of LiteLLM Memory:**
- **Lightweight Implementation**: Custom classes without framework overhead.
- **Direct API Control**: Full control over message formatting and API calls.
- **Flexible Memory Strategies**: Easy to customize and extend memory behavior.
- **Provider Agnostic**: Works with any LiteLLM-supported model.
- **Simple Integration**: Direct integration with existing LiteLLM workflows.
- **Persistence Option**: Manual implementation for disk-based storage.

**Memory types presented in this notebook:**
- **Buffer Memory**: Store and pass all conversation history.
- **Summary Memory**: Periodically summarize the conversation to keep context without full history.
- **Buffer Window Memory**: Keep only the last N messages.
- **Summary Buffer Memory**: Combines buffer and summary - keeps recent messages and summarizes older ones.
- **Persistent Buffer Memory**: Stores conversation history to disk for persistence across sessions.

In [1]:
# Install LiteLLM if not already installed
# !pip install litellm

import litellm

# Example model (local Ollama)
MODEL = 'ollama/llama3.2:1b'

## Buffer Memory
Keeps the entire conversation history and passes it to each LLM call.

In [2]:
class BufferMemory:
    def __init__(self):
        self.messages = []
    
    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
    
    def get_history(self):
        return self.messages.copy()
    
    def clear(self):
        self.messages = []

# Initialize memory
buffer_memory = BufferMemory()

# Function to chat with memory
def chat_with_buffer(user_input):
    buffer_memory.add_message('user', user_input)
    response = litellm.completion(model=MODEL, messages=buffer_memory.get_history())
    ai_response = response.choices[0].message.content
    buffer_memory.add_message('assistant', ai_response)
    return ai_response

# Example usage
print('AI:', chat_with_buffer('Hello, my name is Alice.'))
print('AI:', chat_with_buffer('What is my name?'))
print('History:', buffer_memory.get_history())

## Summary Memory
Summarizes the conversation periodically to reduce token usage while maintaining context.

In [3]:
class SummaryMemory:
    def __init__(self, summary_threshold=10):
        self.messages = []
        self.summary = ''
        self.summary_threshold = summary_threshold
    
    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        if len(self.messages) > self.summary_threshold:
            self._summarize()
    
    def _summarize(self):
        # Summarize the conversation
        conversation_text = '\n'.join([f"{m['role']}: {m['content']}" for m in self.messages])
        prompt = f"Summarize this conversation:\n{conversation_text}"
        summary_response = litellm.completion(model=MODEL, messages=[{'role': 'user', 'content': prompt}])
        self.summary = summary_response.choices[0].message.content
        self.messages = [{'role': 'system', 'content': f'Summary: {self.summary}'}]  # Reset to summary
    
    def get_history(self):
        return self.messages.copy()
    
    def clear(self):
        self.messages = []
        self.summary = ''

# Initialize memory
summary_memory = SummaryMemory(summary_threshold=4)  # Summarize after 4 messages

# Function to chat with summary memory
def chat_with_summary(user_input):
    summary_memory.add_message('user', user_input)
    response = litellm.completion(model=MODEL, messages=summary_memory.get_history())
    ai_response = response.choices[0].message.content
    summary_memory.add_message('assistant', ai_response)
    return ai_response

# Example usage
print('AI:', chat_with_summary('I like pizza.'))
print('AI:', chat_with_summary('What do I like?'))
print('AI:', chat_with_summary('Tell me about my preferences.'))
print('AI:', chat_with_summary('Remind me what we talked about.'))
print('History:', summary_memory.get_history())

## Buffer Window Memory
Keeps only the last N messages to limit context length.

In [4]:
class BufferWindowMemory:
    def __init__(self, window_size=5):
        self.messages = []
        self.window_size = window_size
    
    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        if len(self.messages) > self.window_size:
            self.messages.pop(0)  # Remove oldest
    
    def get_history(self):
        return self.messages.copy()
    
    def clear(self):
        self.messages = []

# Initialize memory
window_memory = BufferWindowMemory(window_size=4)

# Function to chat with window memory
def chat_with_window(user_input):
    window_memory.add_message('user', user_input)
    response = litellm.completion(model=MODEL, messages=window_memory.get_history())
    ai_response = response.choices[0].message.content
    window_memory.add_message('assistant', ai_response)
    return ai_response

# Example usage
print('AI:', chat_with_window('First message.'))
print('AI:', chat_with_window('Second message.'))
print('AI:', chat_with_window('Third message.'))
print('AI:', chat_with_window('Fourth message.'))
print('AI:', chat_with_window('What was the first message?'))  # Should forget first
print('History:', window_memory.get_history())

## Summary Buffer Memory
Combines buffer and summary: keeps recent messages and summarizes older ones.

In [5]:
class SummaryBufferMemory:
    def __init__(self, max_tokens=100, recent_messages=4):
        self.messages = []
        self.summary = ''
        self.max_tokens = max_tokens
        self.recent_messages = recent_messages  # Keep last N messages in full
    
    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        self._check_and_summarize()
    
    def _check_and_summarize(self):
        # Rough token estimation
        total_tokens = sum(len(msg['content'].split()) for msg in self.messages)
        if total_tokens > self.max_tokens and len(self.messages) > self.recent_messages:
            # Keep recent messages, summarize older ones
            recent = self.messages[-self.recent_messages:]
            older = self.messages[:-self.recent_messages]
            
            if older:
                conversation_text = '\n'.join([f"{m['role']}: {m['content']}" for m in older])
                prompt = f"Summarize this conversation concisely:\n{conversation_text}"
                summary_response = litellm.completion(model=MODEL, messages=[{'role': 'user', 'content': prompt}])
                self.summary = summary_response.choices[0].message.content
            
            # Reset messages to summary + recent
            self.messages = [{'role': 'system', 'content': f'Summary: {self.summary}'}] + recent
    
    def get_history(self):
        return self.messages.copy()
    
    def clear(self):
        self.messages = []
        self.summary = ''

# Initialize memory
summary_buffer_memory = SummaryBufferMemory(max_tokens=100, recent_messages=4)

# Function to chat with summary buffer memory
def chat_with_summary_buffer(user_input):
    summary_buffer_memory.add_message('user', user_input)
    response = litellm.completion(model=MODEL, messages=summary_buffer_memory.get_history())
    ai_response = response.choices[0].message.content
    summary_buffer_memory.add_message('assistant', ai_response)
    return ai_response

# Example usage with longer conversation
print('AI:', chat_with_summary_buffer('Tell me a joke.'))
print('AI:', chat_with_summary_buffer('Another one please.'))
print('AI:', chat_with_summary_buffer('One more joke.'))
print('AI:', chat_with_summary_buffer('Tell me a funny story.'))
print('AI:', chat_with_summary_buffer('What jokes have I heard?'))
print('History:', summary_buffer_memory.get_history())

## Persistent Buffer Memory
Stores conversation history to disk for persistence across sessions.

In [6]:
import json
import os

class PersistentBufferMemory:
    def __init__(self, file_path='chat_history.json'):
        self.file_path = file_path
        self.messages = self._load_history()
    
    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})
        self._save_history()
    
    def get_history(self):
        return self.messages.copy()
    
    def clear(self):
        self.messages = []
        self._save_history()
    
    def _load_history(self):
        if os.path.exists(self.file_path):
            with open(self.file_path, 'r') as f:
                return json.load(f)
        return []
    
    def _save_history(self):
        with open(self.file_path, 'w') as f:
            json.dump(self.messages, f)

# Initialize persistent memory
persistent_memory = PersistentBufferMemory()

# Function to chat with persistent memory
def chat_with_persistent(user_input):
    persistent_memory.add_message('user', user_input)
    response = litellm.completion(model=MODEL, messages=persistent_memory.get_history())
    ai_response = response.choices[0].message.content
    persistent_memory.add_message('assistant', ai_response)
    return ai_response

# Example usage
print('AI:', chat_with_persistent('Remember my favorite color is blue.'))
print('AI:', chat_with_persistent('What is my favorite color?'))
print('History:', persistent_memory.get_history())